# Phase 2 — Section 2 Report
## EDA, Data Preprocessing, and Feature Engineering

**Dataset:** Crawled Stack Overflow C++ Questions Dataset  
**Final task:** Semantic similar-question recommendation  

Run `python pipeline.py` before this notebook. The cells below query the SQLite database and inspect the same data that is used by the pipeline.

## 1. Preparation scope

Section 1 handled **ingestion integrity**: source-field validation, unique question IDs, valid tag lists, normalized tables, and timestamp conversion. This section handles **modeling preparation**: EDA, missingness, noisy fields, text cleaning, distribution/outlier checks, and feature engineering. Section 3 packages the same steps into modular, reproducible scripts.

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib
matplotlib.use('Agg')  # Non-interactive backend: works consistently in VS Code and automated runs.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
environment_root = Path(os.environ['PROJECT_ROOT']).resolve() if os.environ.get('PROJECT_ROOT') else None
candidates = [path for path in [environment_root, cwd, *cwd.parents, cwd / 'Phase 2' / 'section1_database'] if path is not None]
PROJECT_ROOT = next(
    (path for path in candidates if (path / 'pipeline.py').exists() and (path / 'requirements.txt').exists()),
    cwd,
)
sys.path.insert(0, str(PROJECT_ROOT))

from scripts.database_connection import get_database_path
from scripts.load_data import load_question_dataframe

DATABASE_PATH = get_database_path()
if not DATABASE_PATH.exists():
    raise FileNotFoundError(f'Database not found: {DATABASE_PATH}. Run python pipeline.py first.')

questions = load_question_dataframe()
questions['creation_at'] = pd.to_datetime(questions['creation_at'], utc=True, errors='coerce')
questions['last_activity_at'] = pd.to_datetime(questions['last_activity_at'], utc=True, errors='coerce')
print(f'Project root: {PROJECT_ROOT}')
print(f'Questions loaded from SQLite: {len(questions):,}')

## 2. Dataset structure

The recommendation input is the title plus the HTML body. Tags and engagement variables are retained as metadata; owner profile and sparse migration fields are deliberately excluded from modeling because they do not represent the technical meaning of a question.

In [ ]:
summary = pd.DataFrame({
    'column': questions.columns,
    'dtype': questions.dtypes.astype(str).values,
    'missing_values': questions.isna().sum().values,
    'unique_values': questions.nunique(dropna=True).values,
})
display(summary)
display(questions.head(3))

## 3. Missing values and data quality

Rows without title or body cannot be used for semantic retrieval and are removed by `preprocess.py`. Optional dates and accepted-answer IDs are not imputed because their absence has a natural meaning (for example, an unedited question has no edit date). Tags are normalized and an empty tag list would be explicitly recorded.

In [ ]:
quality_checks = pd.DataFrame({
    'check': [
        'duplicate question_id', 'missing title', 'missing body_html',
        'blank title', 'blank body_html', 'questions without tags'
    ],
    'count': [
        questions['question_id'].duplicated().sum(),
        questions['title'].isna().sum(),
        questions['body_html'].isna().sum(),
        questions['title'].fillna('').str.strip().eq('').sum(),
        questions['body_html'].fillna('').str.strip().eq('').sum(),
        questions['tags'].isna().sum(),
    ],
})
display(quality_checks)
display(questions.isna().sum().sort_values(ascending=False).to_frame('missing_values'))

## 4. Numeric distributions and outliers

Views, answers, and score measure community engagement rather than semantic similarity. They are highly skewed, so the pipeline keeps their raw versions for auditability but creates `log1p`/signed-log transformations and standardized variants for optional ranking metadata. Raw values are not used as the text-similarity target.

In [ ]:
numeric_columns = ['view_count', 'answer_count', 'score', 'tag_count']
display(questions[numeric_columns].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T)

def iqr_outlier_count(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return int(((series < q1 - 1.5 * iqr) | (series > q3 + 1.5 * iqr)).sum())

outliers = pd.DataFrame({
    'feature': numeric_columns,
    'iqr_outlier_count': [iqr_outlier_count(questions[column]) for column in numeric_columns],
})
display(outliers)

In [ ]:
plot_data = questions.copy()
plot_data['log_view_count'] = np.log1p(plot_data['view_count'].clip(lower=0))
plot_data['signed_log_score'] = np.sign(plot_data['score']) * np.log1p(plot_data['score'].abs())
monthly_questions = plot_data.set_index('creation_at').resample('ME').size()

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes[0, 0].hist(plot_data['log_view_count'], bins=40, color='#377eb8')
axes[0, 0].set(title='log1p(view count)', xlabel='log1p(views)', ylabel='questions')
axes[0, 1].hist(plot_data['signed_log_score'], bins=40, color='#4daf4a')
axes[0, 1].set(title='Signed log score', xlabel='signed log1p(score)', ylabel='questions')
axes[1, 0].hist(plot_data['tag_count'], bins=np.arange(0.5, 6.5, 1), color='#984ea3', rwidth=0.9)
axes[1, 0].set(title='Tags per question', xlabel='tag count', ylabel='questions')
axes[1, 1].plot(monthly_questions.index, monthly_questions.values, color='#e41a1c', linewidth=1)
axes[1, 1].set(title='Question creation over time', xlabel='month', ylabel='questions')
fig.tight_layout()
display(fig)
plt.close(fig)

## 5. Text and tag patterns

The corpus is entirely within the C++ domain, but secondary tags reveal subtopics. This supports a two-stage future retrieval design: semantic similarity over the cleaned document text, optionally constrained or reranked using tags.

In [ ]:
tag_frequency = (
    questions['tags'].fillna('').str.split('|').explode().replace('', pd.NA).dropna()
    .value_counts().rename_axis('tag').reset_index(name='question_count')
)
display(tag_frequency.head(20))

text_lengths = pd.DataFrame({
    'title_characters': questions['title'].fillna('').str.len(),
    'body_html_characters': questions['body_html'].fillna('').str.len(),
})
display(text_lengths.describe(percentiles=[0.01, 0.5, 0.95, 0.99]).T)

## 6. Implemented preprocessing and feature engineering

`scripts/preprocess.py` removes duplicate/invalid text rows, turns HTML into normalized text, normalizes tag lists, parses dates, and records all decisions in `data/reports/preprocessing_report.json`.

`scripts/feature_engineering.py` then creates title/body/document word counts, unique-token ratio, code-block count, creation year/month/age, tag count, log-transformed engagement values, standardized metadata, and TF-IDF unigrams/bigrams. The TF-IDF matrix is the direct feature representation for cosine-similarity recommendation.

In [ ]:
import json

report_dir = PROJECT_ROOT / 'data' / 'reports'
for report_name in ['preprocessing_report.json', 'feature_engineering_report.json']:
    report_path = report_dir / report_name
    if report_path.exists():
        print(f'\n{report_name}')
        display(pd.Series(json.loads(report_path.read_text(encoding='utf-8'))))
    else:
        print(f'{report_name} is not available yet; run python pipeline.py.')

## 7. Section 2 conclusion

The data is ready for direct modeling: every retained row has usable question text, the preprocessing is deterministic, and the feature artefacts are saved. The next stage uses `pipeline.py` to reproduce these results automatically; Phase 3 can use the saved TF-IDF matrix to compute cosine similarity between a new question and historical questions.